In [ ]:
import torch
import pandas as pd
import logging

In [ ]:
data = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/6778e5d1_May27-15-59-58_sd1000_dev_tune0/models/eval/training_124999/5678c19d_Jun01-19-43-34_sd1000_INF_dev/predictions/pred.pt")


In [ ]:
data_dinov2 = torch.load("/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/89d3ad98_May23-13-58-49_sd1000_dev_tune0/models/eval/training_124999/f5385048_Jun01-17-04-04_sd1000_INF_dev/predictions/pred.pt")


In [ ]:
def find_emb_sim_top_k(query_feat: torch.Tensor,
                       tgt_feat: torch.Tensor,
                       k: int,
                       mask: torch.Tensor = None, device="cpu"):

    query_feat, tgt_feat = query_feat.to(device), tgt_feat.to(device)
    if mask is not None: mask = mask.to(device)

    sim = (torch.nn.functional.normalize(query_feat, p=2, dim=1)
        @ torch.nn.functional.normalize(tgt_feat, p=2, dim=1).T)
    if mask is not None: sim[mask.T] = 0

    logging.info(f"Sim matrix shape {sim.shape}")
    return sim.argsort(dim=1, descending=True)[:, :k], sim


In [ ]:
def filter_df_by_slide(data):
    data = pd.DataFrame(data)
    data["slide"] = data["path"].str.extract("(NIO_UCSF_[0-9a-zA-Z]+-[0-9a-zA-Z]+)")
    return data[data["slide"]=="../data/data/cell_viz_slides.csv-5"].reset_index(drop=True)

In [ ]:
def find_nn_matches(data, k):
    query = data.sample(64)
    query_emb = torch.stack(query["embeddings"].tolist())
    slide_emb = torch.stack(data["embeddings"].tolist())
    
    tk, _ = find_emb_sim_top_k(query_emb, slide_emb,  k=12, mask=None)
    
    query["match"] = [[data["path"][i.item()] for i in j] for j in tk]
    
    return query

In [ ]:
def decode_cell_coord(in_str):
    in_patch = [int(i) for i in in_str.split("#")[-1].split("_")]
    patch_coord = [int(i) for i in in_str.split("#")[0].split("-")[-1].split("_")]
    return (patch_coord[0] + patch_coord[2] + in_patch[0]
            , patch_coord[1]*0.9 + patch_coord[3] + in_patch[1])

In [ ]:
from ts2.train.main_cell_inference import SingleCellTempInferenceDataset
from omegaconf import OmegaConf
import yaml
from ts2.data.db_improc import MemmapTileReader
from ts2.data.transforms import HistologyTransform
import io
import base64
import einops
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
from PIL import Image
def im_to_bytestr(image) -> str:
        output = io.BytesIO()
        Image.fromarray(image).save(output, format='JPEG')
        return "data:image/jpeg;base64," + base64.b64encode(
            output.getvalue()).decode()
    

In [ ]:
!ls /nfs/turbo/umms-tocho-snr/data/root_histology_db/srh/UCSF/

In [ ]:
with open("../train/config/chengjia/inference_dinov2_scsrh.yaml") as fd:
    cf = OmegaConf.create(yaml.safe_load(fd))
cf.data.test_dataset.params.slide_instances = "../data/data/cell_viz_slides.csv"

dataset = SingleCellTempInferenceDataset(
     transform=HistologyTransform(**cf.data.xform_params),
    **cf.data.test_dataset.params)

In [ ]:
all_ims = [dataset[i] for i in range(len(dataset))]


In [ ]:
dv2_filtered = filter_df_by_slide(data_dinov2)

In [ ]:
dv2_matches = find_nn_matches(dv2_filtered, k=12)

In [ ]:
dv2_matches

In [ ]:
query